# Clase 10 — Práctica: Dash, callbacks e interactividad

Dos ejercicios para practicar el patrón `@app.callback` sobre los dos datasets que ya conoces: aerolíneas e incendios en Australia.

**Recuerda la limitación técnica:** Dash no corre nativo en Colab. Para probar tu solución:

- **Opción A:** copia el código a un archivo `.py` (puedes usar `%%writefile`) y córrelo en una terminal local con `python archivo.py`.
- **Opción B:** cambia la última línea a `app.run(jupyter_mode='inline')` para verlo embebido en el notebook (Dash 2.11+ lo trae incluido, sin instalar nada adicional).

Ambas opciones están explicadas con código completo en el notebook de teoría (`clase10_dash-interactividad.ipynb`).

In [ ]:
import pandas as pd

# Dataset de aerolíneas (lo usarás en el Ejercicio 1)
airline_data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str})

# Dataset de incendios en Australia (lo usarás en el Ejercicio 2)
wildfire_df = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Historical_Wildfires.csv')
wildfire_df['Month'] = pd.to_datetime(wildfire_df['Date']).dt.month_name()
wildfire_df['Year'] = pd.to_datetime(wildfire_df['Date']).dt.year

airline_data.head()


## Ejercicio 1 — Vuelos por estado destino y aerolínea

Construye un dashboard con el mismo patrón del **Ejemplo 1** de la clase (un campo `dcc.Input` para el año, un `dcc.Graph` que se recalcula con un callback), pero esta vez mostrando un **bar chart del número total de vuelos a cada estado de destino, separado por aerolínea (`Reporting_Airline`)**, para el año introducido.

**Pistas:**

- Filtra `airline_data` por el año igual que en el Ejemplo 1: `df = airline_data[airline_data['Year'] == int(entered_year)]`.
- Agrupa por `DestState` y `Reporting_Airline`, y cuenta el número de vuelos. Una forma cómoda: `df.groupby(['DestState', 'Reporting_Airline'])['Flights'].sum().reset_index()` (la columna `Flights` en este dataset indica el número de vuelos de cada fila; también podrías usar `.size()` si prefieres contar filas).
- Usa `px.bar(..., x='DestState', y='Flights', color='Reporting_Airline', title=...)`.
- El Output del callback sigue siendo `component_property='figure'` sobre un `dcc.Graph(id='bar-plot')` vacío en el layout — igual que `line-plot` en el Ejemplo 1.
- El título de la app puede ser `"Total number of flights to the destination state split by reporting airline"`.


In [ ]:
%%writefile clase10_practica_ejercicio1.py
# TODO: Ejercicio 1 - Vuelos por estado destino y aerolínea
import pandas as pd
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px

airline_data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str})

app = dash.Dash(__name__)

# TODO: define app.layout
# - Título con html.H1 (usa el texto sugerido en las pistas)
# - Un html.Div con "Input Year: " y un dcc.Input(id='input-year', value='2010', type='number')
# - Un dcc.Graph(id='bar-plot') vacío (sin figure=...)
app.layout = html.Div(children=[
    # ... tu código aquí ...
])


# TODO: agrega el decorador @app.callback
# - Output: component_id='bar-plot', component_property='figure'
# - Input: component_id='input-year', component_property='value'
def get_graph(entered_year):
    # TODO:
    # 1. Filtra airline_data por el año recibido (conviértelo a int)
    # 2. Agrupa por DestState y Reporting_Airline, sumando/contando Flights
    # 3. Crea el bar chart con px.bar (x='DestState', color='Reporting_Airline')
    # 4. Return la figura
    pass


if __name__ == '__main__':
    app.run(debug=True)


<details>
<summary>💡 Ver solución</summary>

```python
# Ejercicio 1 - Vuelos por estado destino y aerolínea (solución completa)
import pandas as pd
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px

# Leemos el dataset de aerolíneas
airline_data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str})

app = dash.Dash(__name__)

# Layout: título, input de año, y un dcc.Graph vacío (la figura la rellena el callback)
app.layout = html.Div(children=[
    html.H1('Total number of flights to the destination state split by reporting airline',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 30}),
    html.Div(["Input Year: ", dcc.Input(id='input-year', value='2010',
              type='number', style={'height': '50px', 'font-size': 35}), ],
              style={'font-size': 40}),
    html.Br(),
    html.Br(),
    html.Div(dcc.Graph(id='bar-plot')),
])


@app.callback(Output(component_id='bar-plot', component_property='figure'),
              Input(component_id='input-year', component_property='value'))
def get_graph(entered_year):
    # 1. Filtramos por el año introducido
    df = airline_data[airline_data['Year'] == int(entered_year)]

    # 2. Agrupamos por estado de destino y aerolínea, sumando el número de vuelos
    bar_data = df.groupby(['DestState', 'Reporting_Airline'])['Flights'].sum().reset_index()

    # 3. Bar chart, coloreado por aerolínea
    fig = px.bar(bar_data, x='DestState', y='Flights', color='Reporting_Airline',
                 title='Flights to Destination State, split by Reporting Airline')
    return fig


# Run the app
if __name__ == '__main__':
    app.run(debug=True)
```
</details>


## Ejercicio 2 — Un tercer control: filtrar el dashboard de incendios por mes

Toma el dashboard de incendios en Australia (Ejemplo 3 de la clase de teoría) y añade un **tercer control**: un `dcc.Dropdown` para elegir el **mes**, con las opciones extraídas de `wildfire_df.Month.unique()`.

**Pistas:**

1. Añade el nuevo control al layout, con su propio `id` (por ejemplo `id='month'`).
2. Añade un tercer `Input(component_id='month', component_property='value')` a la lista de `Input` del callback.
3. Añade `input_month` como tercer argumento de la función `reg_year_display`, respetando el mismo orden en que declaraste los tres `Input` (region, year, month).
4. Usa `input_month` para filtrar `y_r_data` un paso más (quédate solo con las filas de ese mes) **antes** de calcular `est_data` y `veg_data`.
5. Corre la aplicación y confirma que, al cambiar cualquiera de los tres controles — región, año o mes — ambos gráficos (pie y bar) se recalculan solos.


In [ ]:
%%writefile clase10_practica_ejercicio2.py
# TODO: Ejercicio 2 - añadir un Dropdown de mes al dashboard de incendios
import pandas as pd
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px

df = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Historical_Wildfires.csv')
df['Month'] = pd.to_datetime(df['Date']).dt.month_name()
df['Year'] = pd.to_datetime(df['Date']).dt.year

app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Australia Wildfire Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 26}),

    html.Div([
        html.Div([
            html.H2('Select Region:', style={'margin-right': '2em'}),
            dcc.RadioItems([{"label": "New South Wales", "value": "NSW"},
                            {"label": "Northern Territory", "value": "NT"},
                            {"label": "Queensland", "value": "QL"},
                            {"label": "South Australia", "value": "SA"},
                            {"label": "Tasmania", "value": "TA"},
                            {"label": "Victoria", "value": "VI"},
                            {"label": "Western Australia", "value": "WA"}],
                           "NSW", id='region', inline=True)]),

        html.Div([
            html.H2('Select Year:', style={'margin-right': '2em'}),
            dcc.Dropdown(df.Year.unique(), value=2005, id='year')
        ]),

        # TODO: añade aquí un tercer html.Div con un html.H2('Select Month:', ...)
        # y un dcc.Dropdown(df.Month.unique(), value='January', id='month')

        html.Div([
            html.Div([], id='plot1'),
            html.Div([], id='plot2')
        ], style={'display': 'flex'}),
    ])
])


# TODO: agrega 'month' como tercer Input del callback
@app.callback([Output(component_id='plot1', component_property='children'),
               Output(component_id='plot2', component_property='children')],
              [Input(component_id='region', component_property='value'),
               Input(component_id='year', component_property='value')])
def reg_year_display(input_region, input_year):  # TODO: añade input_month como tercer argumento
    region_data = df[df['Region'] == input_region]
    y_r_data = region_data[region_data['Year'] == input_year]

    # TODO: filtra y_r_data también por input_month antes de continuar

    est_data = y_r_data.groupby('Month')['Estimated_fire_area'].mean().reset_index()
    fig1 = px.pie(est_data, values='Estimated_fire_area', names='Month',
                  title="{} : Monthly Average Estimated Fire Area in year {}".format(input_region, input_year))

    veg_data = y_r_data.groupby('Month')['Count'].mean().reset_index()
    fig2 = px.bar(veg_data, x='Month', y='Count',
                  title='{} : Average Count of Pixels for Presumed Vegetation Fires in year {}'.format(input_region, input_year))

    return [dcc.Graph(figure=fig1),
            dcc.Graph(figure=fig2)]


if __name__ == '__main__':
    app.run(debug=True)


<details>
<summary>💡 Ver solución</summary>

```python
# Ejercicio 2 - Dashboard de incendios con Region + Year + Month (solución completa)
import pandas as pd
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px

# Leemos el dataset de incendios en Australia
df = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Historical_Wildfires.csv')
df['Month'] = pd.to_datetime(df['Date']).dt.month_name()
df['Year'] = pd.to_datetime(df['Date']).dt.year

app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Australia Wildfire Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 26}),

    html.Div([
        html.Div([
            html.H2('Select Region:', style={'margin-right': '2em'}),
            dcc.RadioItems([{"label": "New South Wales", "value": "NSW"},
                            {"label": "Northern Territory", "value": "NT"},
                            {"label": "Queensland", "value": "QL"},
                            {"label": "South Australia", "value": "SA"},
                            {"label": "Tasmania", "value": "TA"},
                            {"label": "Victoria", "value": "VI"},
                            {"label": "Western Australia", "value": "WA"}],
                           "NSW", id='region', inline=True)]),

        html.Div([
            html.H2('Select Year:', style={'margin-right': '2em'}),
            dcc.Dropdown(df.Year.unique(), value=2005, id='year')
        ]),

        # Nuevo control: Dropdown de mes
        html.Div([
            html.H2('Select Month:', style={'margin-right': '2em'}),
            dcc.Dropdown(df.Month.unique(), value='January', id='month')
        ]),

        html.Div([
            html.Div([], id='plot1'),
            html.Div([], id='plot2')
        ], style={'display': 'flex'}),
    ])
])


# Tercer Input añadido a la lista, en el mismo orden en que se usa en la función
@app.callback([Output(component_id='plot1', component_property='children'),
               Output(component_id='plot2', component_property='children')],
              [Input(component_id='region', component_property='value'),
               Input(component_id='year', component_property='value'),
               Input(component_id='month', component_property='value')])
def reg_year_display(input_region, input_year, input_month):
    # data
    region_data = df[df['Region'] == input_region]
    y_r_data = region_data[region_data['Year'] == input_year]

    # Filtro adicional por mes, antes de calcular los agregados
    y_r_data = y_r_data[y_r_data['Month'] == input_month]

    # Plot one - Monthly Average Estimated Fire Area
    est_data = y_r_data.groupby('Month')['Estimated_fire_area'].mean().reset_index()
    fig1 = px.pie(est_data, values='Estimated_fire_area', names='Month',
                  title="{} : Estimated Fire Area in {} {}".format(input_region, input_month, input_year))

    # Plot two - Average Count of Pixels for Presumed Vegetation Fires
    veg_data = y_r_data.groupby('Month')['Count'].mean().reset_index()
    fig2 = px.bar(veg_data, x='Month', y='Count',
                  title='{} : Average Count of Pixels for Presumed Vegetation Fires in {} {}'.format(
                      input_region, input_month, input_year))

    return [dcc.Graph(figure=fig1),
            dcc.Graph(figure=fig2)]


if __name__ == '__main__':
    app.run(debug=True)
```

Al cambiar región, año o mes, los tres `Input` disparan el mismo callback, que siempre devuelve los dos gráficos ya filtrados por las tres condiciones.
</details>
